# GitLab Projects Export (High-Performance)

Exports projects created in the **last 1 year** to **XLSX + JSON**.

### Performance optimisations
- `created_at` date filter applied **before** any file API calls — old projects are skipped immediately
- Raw file endpoint used instead of base64 encode/decode
- Projects processed in parallel (`MAX_WORKERS` threads)
- Files inside each project fetched in parallel (`FILE_WORKERS` threads)
- `repository_tree(recursive=True, all=True)` — one API call per project for the full file list
- `pd.DataFrame` built once at the end; date parsing done in a single UTC pass

In [ ]:
import getpass
import json
import logging
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone, timedelta
from typing import Any, Dict, List, Optional, Tuple

import gitlab
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger('gitlab_export')
logger.info('Ready.')

## ⚙️ Configuration — edit here

In [ ]:
GITLAB_URL   = 'https://devcloud.ubs.net'
GROUP_PATH   = 'ubs/gwma'
OUTPUT_XLSX  = 'gitlab_projects_export.xlsx'
OUTPUT_JSON  = 'gitlab_projects_export.json'
PER_PAGE     = 100
MAX_WORKERS  = 20   # parallel project threads
FILE_WORKERS = 10   # parallel file-fetch threads per project
CUTOFF_DAYS  = 365  # only process projects created within this many days

SOURCE_EXTENSIONS = ('.jsx', '.tsx', '.js', '.ts')

IMPORT_RE = re.compile(
    r"^[ \t]*import\s+\{[^}]+\}(?:\s*,\s*\{[^}]+\})*\s+from\s+"
    r"['\"](@ubs\\.websdk/[^'\"]+|@uwr[^'\"]+)['\"][^\n]*",
    re.MULTILINE,
)

CUTOFF = datetime.now(timezone.utc) - timedelta(days=CUTOFF_DAYS)
logger.info('Cutoff date: %s', CUTOFF.date())

## 🔐 Authentication

In [ ]:
private_token = getpass.getpass('Enter your GitLab private token: ')
client = gitlab.Gitlab(GITLAB_URL, private_token=private_token)
logger.info('Client ready for %s', GITLAB_URL)

## 🔧 Helper Functions

In [ ]:
def created_within(created_at_str: str) -> bool:
    """True if project was created on or after CUTOFF."""
    try:
        dt = datetime.fromisoformat(created_at_str.replace('Z', '+00:00'))
        return dt >= CUTOFF
    except Exception:
        return True  # include if unparseable


def extract_team_name(web_url: str) -> str:
    """Second-last URL path segment. .../team/project → team"""
    try:
        segs = web_url.rstrip('/').split('//', 1)[-1].split('/')[1:]
        if len(segs) >= 2:
            return segs[-2]
    except Exception:
        pass
    return ''


def _raw_file(project: Any, path: str, ref: str) -> Optional[str]:
    """Fetch raw file content — avoids base64 round-trip."""
    try:
        return project.files.raw(file_path=path, ref=ref).decode('utf-8', errors='replace')
    except Exception:
        return None


def get_package_json_info(project: Any, ref: str) -> Tuple[str, str]:
    content = _raw_file(project, 'package.json', ref)
    if content is None:
        logger.info('  [pkg] not found | %s', project.path_with_namespace)
        return 'No', '-'
    try:
        deps = json.loads(content).get('dependencies', {})
    except json.JSONDecodeError as e:
        logger.warning('  [pkg] bad JSON | %s | %s', project.path_with_namespace, e)
        return 'Yes', 'external'
    uwr = [d for d in deps if d.startswith('@uwr/')]
    if uwr:
        logger.info('  [pkg] internal | %s | %s', project.path_with_namespace, uwr)
        return 'Yes', 'internal'
    logger.info('  [pkg] external | %s', project.path_with_namespace)
    return 'Yes', 'external'


def scan_imports(project: Any, ref: str) -> Tuple[str, str, str]:
    try:
        all_items = project.repository_tree(ref=ref, recursive=True, all=True, per_page=100)
    except Exception as e:
        logger.warning('  [scan] tree error | %s | %s', project.path_with_namespace, e)
        return '', '', ''

    src = [i for i in all_items
           if i.get('type') == 'blob' and i['path'].endswith(SOURCE_EXTENSIONS)]
    if not src:
        return '', '', ''

    logger.info('  [scan] %d source files | %s', len(src), project.path_with_namespace)

    filenames, urls, stmts = [], [], []

    def scan_one(item):
        path = item['path']
        content = _raw_file(project, path, ref)
        if not content:
            return None
        results = []
        for m in IMPORT_RE.finditer(content):
            results.append((
                path.split('/')[-1],
                f"{project.web_url}/-/blob/{ref}/{path}",
                m.group(0).strip(),
            ))
        return results or None

    with ThreadPoolExecutor(max_workers=FILE_WORKERS) as pool:
        for result in pool.map(scan_one, src):
            if result:
                for fn, fu, st in result:
                    filenames.append(fn)
                    urls.append(fu)
                    stmts.append(st)

    if not filenames:
        return '', '', ''

    logger.info('  [scan] %d import(s) | %s', len(filenames), project.path_with_namespace)
    fmt = lambda lst: ''.join(f'[{v}]' for v in lst)
    return fmt(filenames), fmt(urls), fmt(stmts)


def derive_library(import_statements: str) -> str:
    """Deduplicated library label from import_statement string.
    Returns 'uwr', 'websdk', or 'uwr, websdk' (no repeats). Empty string if no imports.
    """
    if not import_statements:
        return ''
    found = []
    if '@uwr/' in import_statements:
        found.append('uwr')
    if '@ubs.websdk/' in import_statements:
        found.append('websdk')
    return ', '.join(found)


def process_one(proj_ref: Any, idx: int, total: int) -> Optional[Dict[str, Any]]:
    """Date-filter first (cheap), then package.json filter, then full fetch+process."""
    created_at_str = getattr(proj_ref, 'created_at', None) or ''
    if created_at_str and not created_within(created_at_str):
        logger.info('[%d/%d] SKIP (old) | %s | created=%s',
                    idx, total, proj_ref.path_with_namespace, created_at_str[:10])
        return None

    logger.info('[%d/%d] → %s', idx, total, proj_ref.path_with_namespace)
    project   = client.projects.get(proj_ref.id)
    namespace = project.namespace or {}
    web_url   = project.web_url
    ref       = project.default_branch or 'main'

    team_name         = extract_team_name(web_url)
    pkg_json, int_ext = get_package_json_info(project, ref)

    # Filter: skip projects with no package.json
    if pkg_json == 'No':
        logger.info('[%d/%d] SKIP (no package.json) | %s', idx, total, project.path_with_namespace)
        return None

    imp_fn, imp_url, imp_st = scan_imports(project, ref)
    library = derive_library(imp_st)

    return {
        'project_id':          project.id,
        'name':                project.name,
        'path':                project.path,
        'path_with_namespace': project.path_with_namespace,
        'group_path':          namespace.get('full_path'),
        'web_url':             web_url,
        'description':         project.description,
        'visibility':          project.visibility,
        'archived':            project.archived,
        'created_at':          project.created_at,
        'last_activity_at':    project.last_activity_at,
        'updated_at':          project.updated_at,
        'default_branch':      ref,
        'forks_count':         getattr(project, 'forks_count', None),
        'star_count':          getattr(project, 'star_count', None),
        'open_issues_count':   getattr(project, 'open_issues_count', None),
        'team_name':           team_name,
        'package_json':        pkg_json,
        'int_ext':             int_ext,
        'component':           '',
        'library':             library,
        'import_filename':     imp_fn,
        'import_file_url':     imp_url,
        'import_statement':    imp_st,
    }


logger.info('All helpers defined.')

## 🚀 Fetch Project List

In [ ]:
logger.info("Fetching project list for group '%s'", GROUP_PATH)
group     = client.groups.get(GROUP_PATH)
proj_refs = group.projects.list(include_subgroups=True, all=True, per_page=PER_PAGE)
total     = len(proj_refs)
logger.info('Total projects in group: %d', total)

## ⚡ Process Projects in Parallel

In [ ]:
rows: List[Dict[str, Any]] = []
done = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    future_map = {
        pool.submit(process_one, ref, idx, total): ref
        for idx, ref in enumerate(proj_refs, start=1)
    }
    for future in as_completed(future_map):
        done += 1
        ref = future_map[future]
        try:
            row = future.result()
            if row:
                rows.append(row)
                logger.info('[%d/%d ✓] %s', done, total, row['path_with_namespace'])
            else:
                logger.info('[%d/%d –] skipped | %s', done, total, ref.path_with_namespace)
        except Exception as e:
            logger.error('[%d/%d ✗] %s | %s', done, total, ref.path_with_namespace, e)

logger.info('Included: %d | Skipped/failed: %d | Total: %d', len(rows), total - len(rows), total)

## 👀 Preview

In [ ]:
df = pd.DataFrame(rows)

for col in ('created_at', 'last_activity_at', 'updated_at'):
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce', utc=True).dt.tz_localize(None)

df.sort_values(['group_path', 'name'], inplace=True, ignore_index=True)

print(f'Projects included : {len(df)}')
print(f'With imports found: {(df["import_filename"] != "").sum()}')
if 'library' in df.columns:
    print('Library breakdown:')
    print(df[df['library'] != '']['library'].value_counts().to_string())
df[['name', 'created_at', 'team_name', 'package_json', 'int_ext', 'library', 'import_filename']].head(10)

## 💾 Write XLSX

In [ ]:
if not rows:
    logger.warning('No rows — skipping XLSX.')
else:
    logger.info('Writing XLSX → %s', OUTPUT_XLSX)
    with pd.ExcelWriter(OUTPUT_XLSX, engine='openpyxl') as writer:
        df.to_excel(writer, index=False, sheet_name='projects')
        ws = writer.sheets['projects']
        for col_cells in ws.columns:
            w = max((len(str(c.value)) if c.value is not None else 0) for c in col_cells)
            ws.column_dimensions[col_cells[0].column_letter].width = min(w + 4, 80)
    print(f'✅ XLSX saved → {OUTPUT_XLSX}')

## 💾 Write JSON

In [ ]:
if not rows:
    logger.warning('No rows — skipping JSON.')
else:
    logger.info('Writing JSON → %s', OUTPUT_JSON)
    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(rows, f, indent=2, default=str, separators=(',', ': '))
    print(f'✅ JSON saved → {OUTPUT_JSON}')